# 1. Install JAX FDM

In [ ]:
# !pip install -q -U jax-fdm compas_notebook  # uncomment to install on colab
import jax_fdm
jax_fdm.__version__

# 2. Import libraries and helper functions

In [ ]:
from math import pi
from math import cos
from math import sin

import numpy as np

from compas.datastructures import Network
from compas.itertools import pairwise

from jax_fdm.datastructures import FDNetwork

from jax_fdm.equilibrium import fdm
from jax_fdm.equilibrium import constrained_fdm

from jax_fdm.visualization import NotebookViewer

In [ ]:
def spiral_points(height_total, height_step, radius, num_cycles):
    """
    Create a list of points along a circular spiral.
    """
    num_points = int(height_total / height_step)
    thetas = np.linspace(0.0, num_cycles*pi, num_points + 1)[:-1]

    points = []
    for i, theta in enumerate(thetas):
        x = radius * cos(theta)
        y = radius * sin(theta)
        points.append([x, y, i * height_step])
    
    return points

# 3. Form-finding

In [ ]:
# create the points that define a spiral
points = spiral_points(height_total=6.0, height_step=0.2, radius=1.0, num_cycles=3.0)

In [ ]:
# create a force density network
spiral = FDNetwork()

for i, point in enumerate(points):
    x, y, z = point
    spiral.add_node(key=i, x=x, y=y, z=z)

num_points = len(points)
for i, j in pairwise(range(num_points)):
    spiral.add_edge(i, j)

In [ ]:
# visualize the network
viewer = NotebookViewer()
viewer.add(spiral, edgewidth=0.05)
viewer.show()

In [ ]:
# add supports
spiral.node_support(0)
spiral.node_support(num_points - 1)

# apply loads
for node in spiral.nodes_free():
    spiral.node_load(node, [0.0, 0.0, -0.1])

# assign edge force densities
for edge in spiral.edges():
    spiral.edge_forcedensity(edge, -4.0)

# visualize
viewer = NotebookViewer()
viewer.add(spiral, edgewidth=0.05, loadscale=2.0)
viewer.show()

In [ ]:
# form-find the spiral
spiral_eq = fdm(spiral)

# visualize the spiral (the input geometry is drawn as a plain network)
viewer = NotebookViewer()
viewer.add(spiral.copy(cls=Network))
viewer.add(spiral_eq, loadscale=3.0, edgewidth=0.05)
viewer.show()

# 4. Constrained form- and load-finding

In [ ]:
from jax_fdm.goals import NodePointGoal

from jax_fdm.losses import Loss
from jax_fdm.losses import SquaredError

from jax_fdm.optimization import SLSQP

from jax_fdm.parameters import EdgeForceDensityParameter
from jax_fdm.parameters import NodeLoadXParameter
from jax_fdm.parameters import NodeLoadYParameter
from jax_fdm.parameters import NodeLoadZParameter


# Define optimization goals
goals = []
for node in spiral.nodes():
    if spiral.is_node_support(node):
        continue
    goal = NodePointGoal(node, target=points[node])
    goals.append(goal)

# Set optimization parameters
parameters = []
# Force densities in the edges
for edge in spiral.edges():
    parameter = EdgeForceDensityParameter(edge, bound_low=None, bound_up=0.0)
    parameters.append(parameter)
# The loads applied at the nodes of the staircase
for node in spiral.nodes():
    parameter = NodeLoadXParameter(node, bound_low=None, bound_up=None)
    parameters.append(parameter)
    parameter = NodeLoadYParameter(node, bound_low=None, bound_up=None)
    parameters.append(parameter)
    parameter = NodeLoadZParameter(node, bound_low=None, bound_up=None)
    parameters.append(parameter)

In [ ]:
# Define loss function to minimize
loss = Loss(SquaredError(goals))

# Solve constrained form- and load-finding problem!
from jax_fdm.optimization import OptimizationRecorder

optimizer = SLSQP()
recorder = OptimizationRecorder(optimizer)

spiral_ceq = constrained_fdm(spiral, loss=loss, optimizer=optimizer, maxiter=1000, parameters=parameters, callback=recorder)

In [ ]:
# Display constrained form
viewer = NotebookViewer()
viewer.add(spiral_ceq, loadscale=1.0, edgewidth=0.05)
viewer.show()

In [ ]:
from jax_fdm.visualization import LossPlotter


loss_plotter = LossPlotter(loss, spiral_ceq)
loss_plotter.plot(recorder.history)
loss_plotter.show()

In [ ]:
for edge in spiral_ceq.edges():
    print(edge, spiral_ceq.edge_force(edge))